<a href="https://colab.research.google.com/github/fborrasumh/bioestadistica/blob/main/cuadernos/Tema_290_Modelos_predictivos_e_IA_en_la_practica_clinica_desarrollo_validacion_y_comunicacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# <font color='steelblue'>Tema 290. Modelos predictivos e inteligencia artificial en la práctica clínica: desarrollo, validación y comunicación</font>

**Bioestadística (1964) · Grado en Medicina · 2.º curso · Curso 2027-2028**

**Módulo 5. Generalización de conclusiones**

**Autor:** Fernando Borrás Rocher · Universidad Miguel Hernández de Elche · ORCID [0000-0002-5519-4573](https://orcid.org/0000-0002-5519-4573)

**Duración:** 50 minutos de clase, más el estudio de los apartados de ampliación

**Práctica:** [unidad 290 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)

**Evaluación:** el examen del Módulo 5 se hace **con Colab**. Practica ejecutando e interpretando cada análisis.

**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es). Todos los pacientes y datos de este cuaderno son sintéticos.

*Antes de trabajar, guarda tu copia: **Archivo → Guardar una copia en Drive**.*

## <font color='steelblue'>Configuración del cuaderno</font>

Ejecuta esta celda al empezar. Instala el módulo de preguntas interactivas y prepara el cuaderno.

In [ ]:
#@title Ejecutar configuración
%pip install -q jupyterquiz==2.9.6.4
from jupyterquiz import display_quiz
print("Cuaderno configurado")

## <font color='steelblue'>Objetivos de aprendizaje</font>

Al terminar esta sesión serás capaz de:

1. Distinguir un modelo **explicativo** (Temas 270 y 280) de un modelo **predictivo**.
2. Separar los datos de desarrollo y de validación, y explicar por qué el rendimiento **aparente** es optimista.
3. Evaluar un modelo por su **discriminación** (AUC) y su **calibración** (gráfico de calibración, puntuación de Brier).
4. Reconocer el deterioro de un modelo al aplicarlo en otra población (**validación externa**) y comprobar su rendimiento por subgrupos.
5. Usar la declaración **TRIPOD+AI** para comunicar y valorar críticamente un modelo predictivo, sea estadístico o de aprendizaje automático.

## <font color='steelblue'>Plan de la sesión</font>

| Bloque | Minutos |
| --- | --- |
| 1. Caso clínico de partida | 5 |
| 2. Conceptos clave | 10 |
| 3. Demostración: desarrollar y validar un modelo | 20 |
| 4. Práctica en la app Bioestadística | 10 |
| 5. Autoevaluación | 5 |

## <font color='steelblue'>1. Caso clínico de partida (5 min)</font>

Una empresa ofrece al hospital un sistema de IA que predice qué pacientes tendrán un evento cardiovascular en 5 años. El folleto dice: «Entrenado con datos de 600 pacientes y 16 variables clínicas. **Exactitud del 89 %. AUC de 0,90**».

Como médico de la comisión que decide si se compra, debes saber hacer las preguntas que el folleto no responde:

1. ¿El AUC se ha medido con los **mismos** pacientes con los que se entrenó, o con pacientes nuevos?
2. ¿Las probabilidades que da son **fiables**? Si dice «30 %», ¿tienen un evento 30 de cada 100 pacientes así?
3. ¿Funciona en **nuestra** población, que es mayor que la del hospital donde se desarrolló?
4. ¿Funciona igual en hombres y mujeres?
5. Si el evento ocurre en el 11 % de los pacientes, ¿qué significa una exactitud del 89 %?

(Pista para la quinta: un «modelo» que diga siempre «no habrá evento» acierta el 89 %.)

## <font color='steelblue'>2. Conceptos clave (10 min)</font>

### <font color='steelblue'>2.1 Explicar o predecir</font>

| | Modelo explicativo | Modelo predictivo |
| --- | --- | --- |
| Pregunta | ¿Se asocia X con Y, ajustando por confusores? | ¿Qué probabilidad de Y tiene este paciente? |
| Lo importante | Coeficientes sin sesgo, con su IC | Que las predicciones sean buenas en pacientes **nuevos** |
| Se evalúa con | Diseño, control de la confusión | Discriminación y calibración en datos de validación |
| Ejemplos | ¿La diabetes aumenta el riesgo? | SCORE2, qSOFA, un sistema de IA de triaje |

Un modelo predictivo no necesita que sus variables sean causas: basta con que predigan. Por eso no se deben interpretar sus coeficientes como efectos.

### <font color='steelblue'>2.2 Rendimiento aparente y validación</font>

Un modelo evaluado con los mismos datos con que se ajustó se ha adaptado también a su **ruido**: su rendimiento **aparente** es optimista (sobreajuste, Tema 280). Hay que medirlo en pacientes que el modelo no ha visto:

| Validación | Cómo | Qué evalúa |
| --- | --- | --- |
| Interna: partición | Reservar una parte de los datos (por ejemplo, un 30 %) | Optimismo, en la misma población |
| Interna: validación cruzada o *bootstrap* | Repetir el ajuste con subconjuntos | Lo mismo, aprovechando mejor los datos |
| **Externa** | Pacientes de otro centro, otro país u otra época | **Transportabilidad**: si funciona en otra población |

Los datos de validación deben quedar **reservados** hasta el final. Si se usan para elegir el modelo, dejan de ser una prueba independiente (ley de Goodhart, Tema 090).

### <font color='steelblue'>2.3 Discriminación y calibración</font>

| Propiedad | Pregunta | Medida |
| --- | --- | --- |
| **Discriminación** | ¿Asigna más riesgo a los que tienen el evento que a los que no? | AUC o estadístico C (Tema 160): 0,5 = azar, 1 = perfecta |
| **Calibración** | ¿Las probabilidades coinciden con las frecuencias observadas? | Gráfico de calibración; cociente observado/esperado |
| Global | Las dos a la vez | Puntuación de Brier: media de (probabilidad − resultado)²; menor es mejor |

Un modelo puede discriminar bien y estar mal calibrado: ordenar bien a los pacientes pero multiplicar por dos todos sus riesgos. Para decidir tratamientos según el riesgo absoluto (Tema 230), la calibración es tan importante como la discriminación.

La **exactitud** (porcentaje de aciertos con un umbral) es una mala medida cuando el evento es poco frecuente.

### <font color='steelblue'>2.4 TRIPOD+AI</font>

La declaración **TRIPOD+AI** (*BMJ*, 2024) es la guía internacional para comunicar estudios de modelos predictivos, tanto de regresión como de aprendizaje automático. Algunos de sus apartados:

| Apartado | Qué debe constar |
| --- | --- |
| Fuente de datos | Origen, fechas, criterios de inclusión |
| Resultado | Definición del evento y horizonte temporal |
| Predictores | Definición y momento de medida |
| Tamaño muestral | Cómo se justificó; número de eventos |
| Datos perdidos | Cuántos y cómo se trataron |
| Desarrollo | Tipo de modelo, ajuste de hiperparámetros |
| Evaluación | Discriminación **y** calibración, con IC, en validación interna y externa |
| Equidad | Rendimiento en subgrupos (sexo, edad, etnia, nivel socioeconómico) |
| Uso previsto | Población, momento de la atención y decisión que apoya |
| Disponibilidad | Código y modelo accesibles |

## <font color='steelblue'>3. Demostración: desarrollar y validar un modelo (20 min)</font>

### <font color='steelblue'>3.1 Los datos: un hospital de desarrollo y uno externo</font>

Simulamos un registro de 600 pacientes del hospital donde se desarrolla el modelo (con 6 predictores reales y 10 variables que no tienen relación con el evento) y 2.000 pacientes de otro hospital, con una población mayor, más hipertensa y con más riesgo por factores que el modelo no incluye (por ejemplo, más dislipemia familiar o peor acceso a la atención primaria).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve

rng = np.random.default_rng(seed=290)
PREDICTORES = ["edad", "mujer", "fumador", "diabetes", "pas", "hdl"] + [f"ruido_{i}" for i in range(10)]

def generar(n, edad_media=60, pas_extra=0, riesgo_no_medido=0.0):
    d = pd.DataFrame({
        "edad": rng.normal(edad_media, 10, n).clip(40, 90),
        "mujer": rng.random(n) < 0.5,
        "fumador": rng.random(n) < 0.25,
        "diabetes": rng.random(n) < 0.18,
    })
    d["pas"] = 105 + 0.5 * d["edad"] + pas_extra + rng.normal(0, 14, n)
    d["hdl"] = 48 + 9 * d["mujer"] + rng.normal(0, 11, n)
    for i in range(10):
        d[f"ruido_{i}"] = rng.normal(0, 1, n)
    logit = (-7.5 + riesgo_no_medido + 0.075 * d["edad"] - 0.5 * d["mujer"] + 0.6 * d["fumador"] + 0.7 * d["diabetes"]
             + 0.02 * (d["pas"] - 120) - 0.02 * (d["hdl"] - 50))
    d["evento"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    return d.astype(float)

desarrollo = generar(600)
externo = generar(2000, edad_media=68, pas_extra=8, riesgo_no_medido=0.5)
print(f"Desarrollo: {len(desarrollo)} pacientes, {desarrollo['evento'].mean():.1%} con evento")
print(f"Externo:    {len(externo)} pacientes, {externo['evento'].mean():.1%} con evento")

### <font color='steelblue'>3.2 La exactitud engaña</font>

In [ ]:
y = desarrollo["evento"]
print(f"«Modelo» que siempre dice «sin evento»: exactitud {(y == 0).mean():.1%}, AUC 0,50")

### <font color='steelblue'>3.3 Rendimiento aparente frente a validación interna</font>

In [ ]:
entreno, prueba = train_test_split(desarrollo, test_size=0.3, random_state=290, stratify=desarrollo["evento"])

modelos = {
    "Regresión logística": LogisticRegression(max_iter=1000),
    "Bosque aleatorio": RandomForestClassifier(n_estimators=300, min_samples_leaf=1, random_state=290),
}
for nombre, modelo in modelos.items():
    modelo.fit(entreno[PREDICTORES], entreno["evento"])
    auc_aparente = roc_auc_score(entreno["evento"], modelo.predict_proba(entreno[PREDICTORES])[:, 1])
    auc_prueba = roc_auc_score(prueba["evento"], modelo.predict_proba(prueba[PREDICTORES])[:, 1])
    print(f"{nombre:>20}: AUC aparente {auc_aparente:.3f} · AUC en pacientes reservados {auc_prueba:.3f}")

El bosque aleatorio, un algoritmo de aprendizaje automático muy flexible, **memoriza** a los pacientes de entrenamiento: su AUC aparente es casi perfecto. En pacientes nuevos rinde como la regresión logística, o peor. Con datos tabulares y pocas variables clínicas, los algoritmos complejos rara vez superan a una regresión logística bien especificada (Christodoulou *et al.*, 2019). Por eso el AUC del folleto, sin decir con qué pacientes se midió, no vale nada.

### <font color='steelblue'>3.4 Validación cruzada</font>

In [ ]:
logistica = LogisticRegression(max_iter=1000)
auc_cv = cross_val_score(logistica, desarrollo[PREDICTORES], desarrollo["evento"], cv=5, scoring="roc_auc")
print("AUC en cada una de las 5 particiones:", auc_cv.round(3))
print(f"AUC medio por validación cruzada: {auc_cv.mean():.3f}")

La validación cruzada divide los datos en 5 partes, entrena con 4 y evalúa con la quinta, rotando. Aprovecha todos los datos y muestra cuánto varía el rendimiento de una partición a otra: con 600 pacientes y unos 60 eventos, el AUC va de menos de 0,60 a más de 0,80 según la partición: la estimación es imprecisa.

### <font color='steelblue'>3.5 Validación externa: discriminación y calibración</font>

In [ ]:
final = modelos["Regresión logística"]  # el modelo desarrollado con los datos de entrenamiento

for nombre, datos in [("Desarrollo (reservados)", prueba), ("Hospital externo", externo)]:
    p = final.predict_proba(datos[PREDICTORES])[:, 1]
    print(f"{nombre:>24}: AUC {roc_auc_score(datos['evento'], p):.3f} · Brier {brier_score_loss(datos['evento'], p):.3f} · "
          f"eventos esperados {p.sum():.0f} · observados {datos['evento'].sum():.0f} (O/E = {datos['evento'].sum() / p.sum():.2f})")

In [ ]:
p_ext = final.predict_proba(externo[PREDICTORES])[:, 1]
observado, predicho = calibration_curve(externo["evento"], p_ext, n_bins=8, strategy="quantile")

plt.figure(figsize=(5.5, 5.5))
plt.plot([0, 0.6], [0, 0.6], color="gray", linestyle="--", label="Calibración perfecta")
plt.plot(predicho, observado, marker="o", label="Modelo en el hospital externo")
plt.xlabel("Probabilidad predicha")
plt.ylabel("Proporción observada de eventos")
plt.title("Gráfico de calibración")
plt.legend()
plt.show()

En el hospital externo:

- La **discriminación** se mantiene parecida (AUC alrededor de 0,70): el modelo sigue ordenando a los pacientes por su riesgo.
- La **calibración** empeora: ocurren unos 480 eventos donde el modelo esperaba unos 420 (O/E = 1,15), y los puntos del gráfico quedan por encima de la diagonal en los riesgos altos. El modelo **infraestima** el riesgo de esta población, porque no incluye factores que aquí son más frecuentes. Usado sin ajustar, dejaría sin tratar a pacientes de alto riesgo.

La solución habitual es **recalibrar** el modelo con datos locales antes de usarlo. Es lo que se hizo con SCORE2, que tiene versiones distintas para regiones europeas de riesgo bajo, moderado, alto y muy alto.

### <font color='steelblue'>3.6 Equidad: rendimiento por subgrupos</font>

In [ ]:
externo["p"] = p_ext
for nombre, grupo in [("Hombres", externo[externo["mujer"] == 0]), ("Mujeres", externo[externo["mujer"] == 1])]:
    print(f"{nombre}: n = {len(grupo)} · eventos {grupo['evento'].sum():.0f} · AUC {roc_auc_score(grupo['evento'], grupo['p']):.3f} · "
          f"O/E {grupo['evento'].sum() / grupo['p'].sum():.2f}")

En conjunto, el error de calibración parecía moderado (O/E = 1,15), pero por sexos es muy distinto: el modelo **infraestima** el riesgo de los hombres (O/E ≈ 1,5) y **sobrestima** el de las mujeres (O/E ≈ 0,85). Los errores se compensaban en el total. TRIPOD+AI pide dar el rendimiento por subgrupos. Un modelo puede funcionar bien en conjunto y mal en un grupo infrarrepresentado en los datos de desarrollo, y reproducir o ampliar desigualdades en la atención.

### <font color='steelblue'>Ampliación: comunicar el riesgo al paciente</font>

In [ ]:
paciente = pd.DataFrame([{**{v: 0.0 for v in PREDICTORES}, "edad": 66, "mujer": 0, "fumador": 1, "diabetes": 0,
                          "pas": 150, "hdl": 42}])
riesgo = final.predict_proba(paciente[PREDICTORES])[0, 1]
print(f"Riesgo estimado a 5 años: {riesgo:.0%}")
print(f"De cada 100 personas como usted, unas {riesgo * 100:.0f} tendrán un evento cardiovascular en 5 años "
      f"y unas {100 - riesgo * 100:.0f} no.")

Las **frecuencias naturales** («de cada 100 personas como usted…») se entienden mejor que los porcentajes y mucho mejor que las odds (Tema 150).

### <font color='steelblue'>Ampliación: errores frecuentes</font>

**Dar el rendimiento aparente** como si fuera el rendimiento real.

**Usar los datos de validación para elegir el modelo** y después presentarlos como prueba independiente.

**Evaluar solo la discriminación**, sin la calibración.

**Usar la exactitud** con eventos poco frecuentes.

**Aplicar un modelo en otra población sin validarlo ni recalibrarlo.**

**Interpretar las variables de un modelo predictivo como causas.**

**Creer que un algoritmo de IA es mejor por ser más complejo.**

## <font color='steelblue'>4. Práctica en la app Bioestadística (10 min)</font>

Abre la **[unidad 290 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)** y completa sus misiones:

1. **Misión básica (sin asistente):** clasificar diez afirmaciones de folletos y resúmenes de modelos de IA como discriminación, calibración, rendimiento aparente o validado.
2. **Misión clínica (con pistas):** desarrollar un modelo con la cohorte cardiovascular, validarlo con una partición reservada y presentar su AUC y su gráfico de calibración.
3. **Misión reto (con asistente):** evaluar con la lista TRIPOD+AI el resumen de un artículo sobre un modelo de IA y redactar las cinco preguntas que harías a la empresa del caso.

Al terminar, exporta el **informe de la unidad**.

## <font color='steelblue'>5. Autoevaluación (5 min)</font>

Responde sin asistente.

In [ ]:
#@title Pregunta interactiva
preguntas = [{'question': 'Un modelo tiene un AUC de 0,97 en los datos de entrenamiento y de 0,71 en pacientes '
              'reservados. ¿Qué ocurre?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Está sobreajustado; su rendimiento real es el de los pacientes '
                         'reservados',
               'correct': True,
               'feedback': 'Correcto: el rendimiento aparente es optimista.'},
              {'answer': 'Los pacientes reservados están mal elegidos',
               'correct': False,
               'feedback': 'La diferencia es la señal típica del sobreajuste.'},
              {'answer': 'Su rendimiento real es la media, 0,84',
               'correct': False,
               'feedback': 'El que cuenta es el de los datos no usados en el ajuste.'},
              {'answer': 'Es un modelo excelente',
               'correct': False,
               'feedback': 'Solo lo es con los pacientes que ya conoce.'}]},
 {'question': 'Un modelo discrimina bien en otro hospital (AUC 0,80) pero predice 50 eventos donde '
              'ocurren 100. ¿Qué le pasa?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Está mal calibrado: infraestima el riesgo en esa población',
               'correct': True,
               'feedback': 'Correcto: O/E = 2; conviene recalibrarlo.'},
              {'answer': 'Discrimina mal',
               'correct': False,
               'feedback': 'Un AUC de 0,80 es una discriminación aceptable.'},
              {'answer': 'Está perfectamente validado',
               'correct': False,
               'feedback': 'La calibración es deficiente.'},
              {'answer': 'Sobrestima el riesgo',
               'correct': False,
               'feedback': 'Predice menos eventos de los que ocurren.'}]},
 {'question': 'El evento ocurre en el 5 % de los pacientes y un modelo tiene una exactitud del 95 '
              '%. ¿Qué concluyes?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Nada: decir siempre «sin evento» da también un 95 % de exactitud',
               'correct': True,
               'feedback': 'Correcto: con eventos raros la exactitud no sirve; hacen falta AUC y '
                           'calibración.'},
              {'answer': 'Que es un modelo excelente',
               'correct': False,
               'feedback': 'La exactitud no informa con eventos raros.'},
              {'answer': 'Que su AUC es 0,95',
               'correct': False,
               'feedback': 'Exactitud y AUC son medidas distintas.'},
              {'answer': 'Que está bien calibrado',
               'correct': False,
               'feedback': 'La exactitud no mide la calibración.'}]},
 {'question': '¿Qué aporta la validación externa que no aporta la interna?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Comprobar si el modelo funciona en otra población, otro centro u otra '
                         'época',
               'correct': True,
               'feedback': 'Correcto: evalúa la transportabilidad.'},
              {'answer': 'Más pacientes para entrenar',
               'correct': False,
               'feedback': 'Los datos externos no se usan para entrenar.'},
              {'answer': 'Un AUC siempre mayor', 'correct': False, 'feedback': 'Suele ser menor.'},
              {'answer': 'Elimina la necesidad de calibrar',
               'correct': False,
               'feedback': 'Precisamente detecta la necesidad de recalibrar.'}]},
 {'question': 'Según TRIPOD+AI, ¿qué debe comunicarse sobre el rendimiento de un modelo?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Discriminación y calibración con IC, en validación, y por subgrupos '
                         'relevantes',
               'correct': True,
               'feedback': 'Correcto.'},
              {'answer': 'Solo el AUC aparente',
               'correct': False,
               'feedback': 'El rendimiento aparente es optimista.'},
              {'answer': 'Solo la exactitud global',
               'correct': False,
               'feedback': 'Es una medida insuficiente.'},
              {'answer': 'Solo los coeficientes del modelo',
               'correct': False,
               'feedback': 'En un modelo predictivo importa el rendimiento.'}]}]
display_quiz(preguntas)

## <font color='steelblue'>Ideas clave</font>

- Un modelo predictivo se juzga por cómo predice en pacientes que no ha visto.
- El rendimiento aparente es optimista; los datos de validación se reservan hasta el final.
- Hay que evaluar la discriminación (AUC) y la calibración; la exactitud engaña con eventos poco frecuentes.
- Al cambiar de población, la calibración suele empeorar: valida externamente y recalibra.
- TRIPOD+AI es la guía para comunicar y valorar modelos predictivos, también los de IA.

## <font color='steelblue'>Glosario</font>

| Término | Definición |
| --- | --- |
| Modelo predictivo | Modelo cuyo objetivo es estimar la probabilidad de un resultado en pacientes individuales |
| Rendimiento aparente | Rendimiento medido con los mismos datos con los que se ajustó el modelo |
| Validación interna | Evaluación con datos de la misma población no usados en el ajuste |
| Validación cruzada | Validación interna que rota las particiones de entrenamiento y evaluación |
| Validación externa | Evaluación en una población distinta de la de desarrollo |
| Discriminación | Capacidad de asignar más riesgo a quienes presentan el evento |
| Calibración | Concordancia entre las probabilidades predichas y las frecuencias observadas |
| Puntuación de Brier | Media del cuadrado de la diferencia entre probabilidad predicha y resultado |
| Recalibración | Ajuste de un modelo existente a una nueva población |
| Bosque aleatorio | Algoritmo de aprendizaje automático que combina muchos árboles de decisión |
| TRIPOD+AI | Guía para comunicar estudios de modelos predictivos, incluidos los de IA |

## <font color='steelblue'>Para saber más</font>

- Collins GS, *et al.* TRIPOD+AI statement: updated guidance for reporting clinical prediction models that use regression or machine learning methods. *BMJ*. 2024;385:e078378.
- Van Calster B, *et al.* Calibration: the Achilles heel of predictive analytics. *BMC Med*. 2019;17(1):230.
- Christodoulou E, *et al.* A systematic review shows no performance benefit of machine learning over logistic regression for clinical prediction models. *J Clin Epidemiol*. 2019;110:12-22.

---
*Tema 290 · Bioestadística (1964) · Grado en Medicina · Universidad Miguel Hernández de Elche · Fernando Borrás Rocher · CC BY 4.0*